# Guru Stock Screener — run on Google Colab

Runs the screener on Colab's Linux VM and shows the web UI **right inside this notebook**. Works from any browser, including **iPad Safari** — no local install, and (unlike an iPad on its own) Colab can pull live SEC data.

**How to use:** `Runtime → Run all`, or run each cell top to bottom. The app appears at the bottom.

Personal use only — not investment advice.

## 1. Get the code and install dependencies

In [ ]:
!git clone https://github.com/dkgoal/learning-R.git
%cd learning-R
!git checkout claude/guru-stock-screener-3gn684
!pip install -q -r requirements.txt

## 2. (Optional) Persist data to Google Drive

Colab VMs are wiped when the runtime disconnects (~90 min idle), which clears the local cache. **Skip this for demo data.** For live data you want to keep, uncomment and run this cell **before** the next one — it stores config + cache in your Drive via the `GURU_HOME` variable the app already honors.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["GURU_HOME"] = "/content/drive/MyDrive/guru-screener"
print("Config + cache will persist at:", os.environ["GURU_HOME"])

## 3. Load data

**Demo** data is instant and needs no network. For **real** data, first edit `config/settings.yaml` — set `app.sec_user_agent` to your name + email (SEC requires it) — then use the `refresh` line instead of `seed`.

In [ ]:
!python -m guru_screener seed
# !python -m guru_screener refresh     # live SEC filings + prices (slower)

## 4. Launch the app

Starts the web server in the background and embeds the UI below. Colab proxies the port to your browser, so only **you** (in your authenticated Colab session) can reach it — safe without a login. On iPad the inline frame is the most reliable; the `serve_kernel_port_as_window` line opens it in a full tab if your browser allows popups.

In [ ]:
import threading, time, socket
from guru_screener.web.app import create_app
from guru_screener.config import load_config

def _free_port():
    s = socket.socket(); s.bind(("127.0.0.1", 0))
    p = s.getsockname()[1]; s.close(); return p

# Safe to re-run: reuse the server started earlier instead of binding twice.
# (A second bind on the same port is what raises "Address already in use".)
if not globals().get("_GURU_PORT"):
    _GURU_PORT = _free_port()
    _app = create_app(load_config())
    threading.Thread(
        target=lambda: _app.run(host="127.0.0.1", port=_GURU_PORT,
                                use_reloader=False),
        daemon=True,
    ).start()
    time.sleep(2)  # give the server a moment to start

from google.colab import output
output.serve_kernel_port_as_iframe(_GURU_PORT, path="/", height="800")
# output.serve_kernel_port_as_window(_GURU_PORT)   # opens a full tab instead

---
Tip: after the app loads, use the **Screener**, **Managers**, and **Backtest** tabs at the top. The backtest is deliberately labelled *approximate — not point-in-time*.